# OmniVoice Studio on Google Colab

Run the full [OmniVoice Studio](https://github.com/debpalash/OmniVoice-Studio) app — voice cloning, voice design, video dubbing, and TTS in 646 languages — on a free Colab GPU, web UI included.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (the notebook also works on CPU, but generation is much slower).

Run the cells top to bottom. Every cell is idempotent — re-running after a hiccup is always safe.

**How this notebook works (and why):**

| Piece | Approach | Why |
|---|---|---|
| Backend install | `uv pip install --system .` | Mirrors the official Docker image: installs into Colab's Python and keeps Colab's preinstalled CUDA PyTorch when it satisfies the requirements, instead of re-downloading multi-GB torch wheels into a venv. |
| Web UI | Built in-notebook with `bun` (~2 min, once per session) | Official releases ship desktop installers only — there is no prebuilt standalone web bundle to download. The backend serves the built SPA itself from `frontend/dist`. |
| Opening the app | Colab's built-in kernel port proxy | No third-party tunnel binaries; the URL is private to your Google session. (A `cloudflared` public-URL alternative is documented in the launch cell.) |

Issues with this notebook are OmniVoice issues — report them at [debpalash/OmniVoice-Studio/issues](https://github.com/debpalash/OmniVoice-Studio/issues).


In [ ]:
# ── 1. GPU check ────────────────────────────────────────────────────────────
# Confirms the runtime has an NVIDIA GPU. Everything still works on CPU, but
# a single sentence can take minutes instead of seconds.
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    print("GPU detected — you're good to go.")
else:
    print("=" * 74)
    print("WARNING: no NVIDIA GPU in this runtime.")
    print("Fix: menu bar -> Runtime -> Change runtime type -> T4 GPU,")
    print("then re-run this notebook from the top.")
    print("(Continuing anyway works, but generation will be very slow on CPU.)")
    print("=" * 74)


In [ ]:
# ── 2. Clone + install (idempotent; first run ~5-8 min) ─────────────────────
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/debpalash/OmniVoice-Studio"
REPO_DIR = "/content/OmniVoice-Studio"
BUN = "/root/.bun/bin/bun"

# Generous network timeouts: Colab -> PyPI is usually fast, but model/wheel
# CDNs occasionally stall and uv's default timeout is short.
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    """Stream a command's output; stop the cell with an actionable message on failure."""
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    rc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode
    if rc != 0:
        raise SystemExit(
            f"\nFAILED: {what or shown} (exit code {rc}).\n"
            "Scroll up for the underlying error. Re-running this cell is safe.\n"
            "Still stuck? Open an issue with the output above:\n"
            "  https://github.com/debpalash/OmniVoice-Studio/issues"
        )

# 2a. Clone the repo (reused if already present)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo already present at {REPO_DIR} — reusing it.")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

# 2b. System packages (ffmpeg: audio/video processing; libsndfile1: soundfile)
run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")

# 2c. bun — builds the web UI (see the intro cell for why we build it here)
if not os.path.exists(BUN):
    run("curl -fsSL https://bun.sh/install | bash", what="bun installer")
os.environ["PATH"] = os.path.dirname(BUN) + os.pathsep + os.environ["PATH"]
run([BUN, "--version"], what="bun version check")

# 2d. Build the frontend (skipped once frontend/dist/index.html exists)
dist_index = os.path.join(REPO_DIR, "frontend", "dist", "index.html")
if os.path.exists(dist_index):
    print("frontend/dist already built — skipping.")
else:
    # The repo is a bun workspace: install at the repo root, build in frontend/.
    run([BUN, "install", "--frozen-lockfile"], cwd=REPO_DIR, what="bun install (workspace)")
    run([BUN, "run", "--cwd", "frontend", "build"], cwd=REPO_DIR, what="frontend build (vite)")
    if not os.path.exists(dist_index):
        raise SystemExit(
            "The frontend build finished but frontend/dist/index.html is missing —\n"
            "scroll up for vite errors, then re-run this cell."
        )

# 2e. Backend dependencies — same command the official Docker image uses.
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
run(["uv", "pip", "install", "--system", "--no-cache", "."],
    cwd=REPO_DIR, what="backend install (uv pip install --system .)")

# 2f. cuDNN 8 side-install for CTranslate2 (WhisperX ASR on GPU). PyTorch 2.8+
# ships cuDNN 9; scripts/setup.py places cuDNN 8 libs where the backend
# preloads them from (<repo>/.venv/.../cudnn8_compat). We create the .venv
# directory only so the script has its expected target — no actual venv is
# used on Colab. Non-fatal: without it, transcription falls back to the
# torch-native Whisper backend automatically.
os.makedirs(os.path.join(REPO_DIR, ".venv"), exist_ok=True)
run([sys.executable, os.path.join(REPO_DIR, "scripts", "setup.py")], what="cuDNN 8 compat setup")

# 2g. Sanity check in a fresh interpreter (this kernel may hold a stale torch)
run([sys.executable, "-c",
     "import torch, torchaudio, uvicorn, fastapi; "
     "print(f'Install OK - torch {torch.__version__}, "
     "CUDA available: {torch.cuda.is_available()}')"],
    what="import sanity check")


In [ ]:
# ── 3. (Optional) Hugging Face token ────────────────────────────────────────
# Only needed for GATED models — e.g. pyannote speaker diarization, used by
# video dubbing for multi-speaker detection. TTS, voice cloning, and voice
# design all work WITHOUT a token, so feel free to skip this cell.
#
# To use one: accept the model terms at
#   https://huggingface.co/pyannote/speaker-diarization-3.1
# then add a Colab Secret named HF_TOKEN (key icon in the left sidebar),
# toggle "Notebook access" ON, and re-run this cell.
import os

try:
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _token
    print("HF_TOKEN loaded from Colab Secrets — gated models (diarization) enabled.")
except Exception:
    print("No HF_TOKEN Colab Secret found — skipping. (Everything except gated "
          "diarization models works without it.)")


In [ ]:
# ── 4. (Optional, recommended) Pre-download the default voice model ─────────
# The default TTS engine fetches k2-fsa/OmniVoice (a few GB) on its very first
# generation. Downloading it up front makes the first request fast instead of
# a several-minute stall. Safe to re-run: already-downloaded files are reused.
from huggingface_hub import snapshot_download

path = snapshot_download("k2-fsa/OmniVoice")
print("Default TTS model cached at:", path)


In [ ]:
# ── 5. Launch the backend ───────────────────────────────────────────────────
# Starts uvicorn on port 3900 and waits for /health. Re-running this cell when
# the backend is already up just reports its status.
import json
import os
import subprocess
import sys
import time
import urllib.request

REPO_DIR = "/content/OmniVoice-Studio"
PORT = 3900
LOG_PATH = "/content/omnivoice_backend.log"
HEALTH_URL = f"http://127.0.0.1:{PORT}/health"

def health():
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if info:
    print(f"Backend already running — {info}")
else:
    # Clear any half-dead process from a previous run of this cell.
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    time.sleep(1)

    env = os.environ.copy()
    # Headless-server deployment, same as the official Docker image: relaxes
    # the desktop-only loopback origin gate so the proxied browser session can
    # reach the settings/system routes.
    env["OMNIVOICE_SERVER_MODE"] = "1"
    # Voices, projects, and generated audio live here. Ephemeral! See the
    # troubleshooting cell for persisting it to Google Drive.
    env["OMNIVOICE_DATA_DIR"] = "/content/omnivoice_data"
    env["PYTHONUNBUFFERED"] = "1"

    log = open(LOG_PATH, "ab")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app",
         "--app-dir", "backend", "--host", "127.0.0.1", "--port", str(PORT)],
        cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    print(f"Backend starting (PID {proc.pid}); log: {LOG_PATH}")

    deadline = time.time() + 300
    while time.time() < deadline:
        if proc.poll() is not None:
            break  # process died — report below
        info = health()
        if info:
            break
        print(".", end="", flush=True)
        time.sleep(3)
    print()

    if info:
        print(f"Backend is up — {info}")
        if "cuda" not in str(info.get("device", "")):
            print("NOTE: device is not CUDA — generation will be slow. "
                  "See cell 1 to enable the GPU runtime.")
    else:
        try:
            with open(LOG_PATH, "r", errors="replace") as f:
                tail = "".join(f.readlines()[-40:])
        except OSError:
            tail = "(no log file found)"
        raise SystemExit(
            "Backend did not become healthy within 5 minutes.\n"
            f"--- last lines of {LOG_PATH} ---\n{tail}\n"
            "--- end of log ---\n"
            "Fix the error above (usually a missing dependency: re-run cell 2),\n"
            "then re-run this cell. Still stuck? Attach the log to an issue:\n"
            "  https://github.com/debpalash/OmniVoice-Studio/issues"
        )


In [ ]:
# ── 6. Open the app ─────────────────────────────────────────────────────────
# Serves localhost:3900 to YOUR browser through Colab's built-in kernel port
# proxy — authenticated to your Google session, no third-party tunnel binary.
# A new browser tab opens with the full OmniVoice Studio UI (allow pop-ups
# for colab.research.google.com if nothing appears).
#
# Want a PUBLIC URL instead (e.g. to open the app on your phone)? Cloudflare's
# quick tunnel works well — run in a new cell:
#   !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
#   !chmod +x /usr/local/bin/cloudflared
#   !nohup cloudflared tunnel --url http://127.0.0.1:3900 --no-autoupdate > /content/cloudflared.log 2>&1 &
#   !sleep 5 && grep -o "https://.*trycloudflare.com" /content/cloudflared.log | head -1
# Anyone with that URL can reach your session — set a share PIN in the app's
# Settings first.
from google.colab import output

output.serve_kernel_port_as_window(3900)
print("If no tab opened, click the link above (and allow pop-ups).")


In [ ]:
# ── 7. Smoke test: generate speech through the API ──────────────────────────
# Proves the whole stack end-to-end without touching the UI: /health, then a
# real TTS generation played inline. POST /generate returns the WAV directly.
import json
import urllib.error
import urllib.parse
import urllib.request

BASE = "http://127.0.0.1:3900"

with urllib.request.urlopen(f"{BASE}/health", timeout=10) as r:
    print("GET /health ->", json.load(r))

data = urllib.parse.urlencode({
    "text": "OmniVoice Studio is up and running on Google Colab.",
}).encode()
print("Generating... (first-ever generation loads the model — if you skipped "
      "cell 4 it also downloads it, which can take several minutes)")
try:
    with urllib.request.urlopen(
        urllib.request.Request(f"{BASE}/generate", data=data), timeout=1800
    ) as r:
        wav = r.read()
        print(f"OK — {len(wav)} bytes, duration {r.headers.get('X-Audio-Duration')}s, "
              f"generated in {r.headers.get('X-Gen-Time')}s (seed {r.headers.get('X-Seed')})")
except urllib.error.HTTPError as e:
    detail = e.read().decode("utf-8", errors="replace")
    raise SystemExit(
        f"Generation failed: HTTP {e.code}\n{detail}\n"
        "Check /content/omnivoice_backend.log for the full trace; on a fresh\n"
        "session the usual cause is an interrupted model download — re-run\n"
        "cell 4, then this cell."
    )

OUT = "/content/omnivoice_smoke.wav"
with open(OUT, "wb") as f:
    f.write(wav)

from IPython.display import Audio, display
display(Audio(OUT))


## Troubleshooting

**"device": "cpu" in /health, or generation is crawling** — the runtime has no GPU. `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Colab's free tier sometimes has no GPUs available; trying again later usually works.

**First generation takes minutes** — that's the default model (a few GB) downloading and loading. Run cell 4 to front-load the download; after the first generation the model stays warm and requests take seconds.

**Session limits (free tier)** — Colab disconnects idle sessions and caps total runtime (up to ~12 h, often less). When the VM is recycled, **everything under `/content` is deleted**: the repo, models, and your `omnivoice_data` (voices, projects, generated audio). To keep your data across sessions, mount Google Drive and point the data dir at it *before* running cell 5:

```python
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['OMNIVOICE_DATA_DIR'] = '/content/drive/MyDrive/omnivoice_data'
```

**VRAM** — a T4 has 16 GB: plenty for TTS, cloning, and voice design. Full dubbing pipelines (separation + transcription + diarization + TTS) run closer to the limit; the app automatically evicts idle engines, but very long videos may need shorter chunks.

**UI tab is blank or errors out** — re-run cell 5 (it verifies `/health` without restarting a healthy backend), then cell 6. Pop-ups must be allowed for `colab.research.google.com`.

**Backend logs** — `/content/omnivoice_backend.log`. The launch and smoke-test cells print the relevant tail on failure; attach the full file when reporting an issue.

**Reporting issues** — this notebook is maintained in the main repo: [debpalash/OmniVoice-Studio/issues](https://github.com/debpalash/OmniVoice-Studio/issues).
